# Comparable-sales selection ablation

Test **both proposed fixes independently** while keeping the original, validated code path. Feature columns, model targets, time features and the 47-term binary vocabulary are identical across candidates.

| Experiment | Name | What changes? |
| --- | --- | --- |
| A | `baseline` | Original closest-128 → age filter → similarity-ranked 32 |
| B | `age_first` | Filter sales aged over 730 days **before** nearest-128 |
| C | `type_quota` | Separate same-property-type candidate search; reserve up to 8 same-type comparables within final 32 (if available) |
| D | `both` | Combine B and C |

The last three use exactly the same 14 comparable feature names, including `comp_log_price`. No model library changes or retuning are required to run the ablation. The original `Xpt_spatiotemporal__selected_binary` is untouched.

**Research caution:** These are hypotheses, not measured performance improvements. The eight-property quota is one fixed initial setting, not an optimized hyperparameter. Distant same-type comps can have tiny similarity weights, so inspect `comp_same_type_share` alongside the scores. Subgroup evidence for co-ops is particularly relevant.

Start with **one earlier development month** to avoid 12 large model fits before knowing whether the modifications work. If promising, evaluate the other earlier folds, and only then make a small number of frozen December–May confirmations.

Run from the project root after `git pull origin main`. No additional Python dependencies are needed.

In [ ]:
# Locate the checkout from Jupyter/VS Code, including nested notebook folders.
from pathlib import Path
import os
import sys
_start = Path(os.environ.get("DATA_DASH_ROOT", Path.cwd())).expanduser().resolve()
ROOT = next((p for p in (_start, *_start.parents)
             if (p / "data.py").is_file() and (p / "models").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Set DATA_DASH_ROOT to your Data Dash checkout before running this notebook.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
import pandas as pd
from IPython.display import display

from comparisons.comparable_benchmark import run_comparable_benchmark, COMP_VARIANTS

BACKEND = "xgboost"
TRACK = "off_market"
MONTHS = ["2025-10"]  # quick first test; later extend to Sep, Oct, Nov
VARIANTS = list(COMP_VARIANTS)

# Copied from the current model_run.ipynb XGB_MODEL_SPECS as of implementation.
# Confirm these still match YOUR best 7.82% XGBoost Binary BoW run.
XGB_PARAMS = {
    "n_estimators": 700,
    "learning_rate": 0.09522271657253033,
    "max_depth": 8,
    "min_child_weight": 16.684619386293466,
    "reg_lambda": 19.50331236061207,
    "reg_alpha": 0.0,
    "subsample": 0.9804685587557792,
    "colsample_bytree": 0.8994749947027713,
    "tree_method": "hist",
    "grow_policy": "depthwise",
    "device": "cpu",
    "enable_categorical": True,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0,
}
PARAMS = XGB_PARAMS
print("Model:", BACKEND, "| Track:", TRACK)
print("Variants:", VARIANTS)
print("Months:", MONTHS)

In [ ]:
summary, monthly, comparison_rows = run_comparable_benchmark(
    backend=BACKEND,
    track=TRACK,
    text_feature="Xpt_spatiotemporal__selected_binary",
    params=PARAMS,
    months=MONTHS,
    variants=VARIANTS,
)
display(summary)
display(monthly)

In [ ]:
# Compare comparable-sales quality, not just aggregate scores.
display(
    comparison_rows.groupby("variant")[
        ["comp_count", "comp_effective_n", "comp_same_type_share",
         "comp_nearest_km", "comp_mean_age_days"]
    ].median().round(3)
)

# Relative to the unchanged baseline: positive values mean a correction
# increased the estimated comparable-sale log value for that property.
aligned = comparison_rows.pivot(
    index=["month", "ml_num"], columns="variant",
    values=["comp_log_price", "predicted_price", "abs_pct_error"],
)
for variant in ["age_first", "type_quota", "both"]:
    if variant not in aligned["comp_log_price"]:
        continue
    differences = (
        aligned["comp_log_price"][variant] -
        aligned["comp_log_price"]["baseline"]
    )
    print(
        variant,
        "| changed valid comp values:",
        int((differences.abs() > 1e-6).sum()),
        "| median abs log-comp shift:",
        float(differences.abs().median()),
    )

# Minority type investigation: inspect the improvement for co-ops separately,
# but consider small subgroup sample sizes before interpreting differences.
co_op = comparison_rows[
    comparison_rows.house_type.astype(str).str.contains(
        "co-op|coop|cooperative", case=False, regex=True
    )
]
if not co_op.empty:
    display(
        co_op.groupby("variant").agg(
            homes=("ml_num", "size"),
            mdape=("abs_pct_error", "median"),
            mape=("abs_pct_error", "mean"),
            median_same_type=("comp_same_type_share", "median"),
        ).round(3)
    )
else:
    print("No co-ops in this validation cohort.")

print("Paired rows saved under outputs/comparable_benchmark/")

## Integration into the existing model pipeline

Once you have frozen the best candidate from **development** folds, change the single feature-name string in `model_run.ipynb` for a December 2025–May 2026 confirmation:

```python
# In the existing XGB_MODEL_SPECS tuple, change ONLY the feature name.
# Keep the existing tuned parameter dictionary unchanged.
(
    XGBoostRegressor,
    "Xpt_spatiotemporal__selected_binary__comp_age_first",
    { ... your existing exact tuned parameters ... },
)
```

Alternatives:
- Original: `Xpt_spatiotemporal__selected_binary`
- Age-first only: `Xpt_spatiotemporal__selected_binary__comp_age_first`
- Same-type quota only: `Xpt_spatiotemporal__selected_binary__comp_type_quota`
- Both: `Xpt_spatiotemporal__selected_binary__comp_both`

The list-aware ratio model also recognizes the same feature names; its `comp_vs_list_log` is recomputed from the chosen comparable estimate. Use your existing `custom_make_model` without modifications.

When rerunning a Jupyter notebook after `git pull`, **restart the kernel** to reload feature classes. Run focused checks first:

```powershell
uv run python -m pytest tests/test_comparable_ablation.py -q
```

Evaluate paired per-property predictions and the fixed-origin horizon as needed. Do not claim an improvement until the comparison finishes.